In [1]:
import pandas as pd
import geopandas as gpd

# Colunas essenciais para criar o log de eventos
COL_VEICULO = 'COD'
COL_LINHA = 'CODIGOLINHA'
COL_SENTIDO = 'SENT'
COL_TIMESTAMP_API = 'request_timestamp' # Usar este é mais confiável que o 'REFRESH'
COL_LAT = 'LAT'
COL_LON = 'LON'

# Colunas padrão do PM4Py
CASE_ID = 'case:concept:name'
ACTIVITY = 'concept:name'
TIMESTAMP = 'time:timestamp'


def clean_dataframe(df_raw):
    print('comeocu')
    """
    Limpa o DataFrame bruto, converte tipos e ordena os dados.
    """
    # 1. Copiar para evitar aviso de 'SettingWithCopyWarning'
    df = df_raw.copy()

    # 2. Converter o timestamp para datetime (crucial!)
    df[COL_TIMESTAMP_API] = pd.to_datetime(df[COL_TIMESTAMP_API])
    
    # 3. Ordenar os dados
    # Garante que os pings de cada ônibus estejam em ordem cronológica
    df = df.sort_values(by=[COL_VEICULO, COL_TIMESTAMP_API])
    
    # 4. Lidar com dados ausentes (se houver)
    # Para o Case ID, precisamos preencher valores nulos em 'LINHA' e 'SENTIDO'
    # 'ffill' (forward fill) preenche com o último valor válido
    df[COL_LINHA] = df[COL_LINHA].ffill()
    df[COL_SENTIDO] = df[COL_SENTIDO].ffill()
    
    print("DataFrame limpo e ordenado.")
    return df

def create_case_id(df):
    """
    Cria um 'Case ID' único para cada viagem, quebrando o trajeto
    com base em mudanças de linha ou sentido.
    
    Esta é a lógica de "quebra de trajeto" que discutimos.
    """
    if df.empty:
        return df

    # 1. Identificar as "quebras" (teleportes)
    # Compara a linha/sentido atual com a linha/sentido anterior (.shift(1))
    mudou_linha = df[COL_LINHA] != df[COL_LINHA].shift(1)
    mudou_sentido = df[COL_SENTIDO] != df[COL_SENTIDO].shift(1)
    
    # Uma "quebra" acontece se a linha MUDOU OU o sentido MUDOU
    quebra_de_viagem = mudou_linha | mudou_sentido

    # 2. Criar o grupo de viagem
    # .cumsum() em um booleano (True=1, False=0) cria um ID de grupo.
    # Ex: F, F, T, F, F, T, F -> 0, 0, 1, 1, 1, 2, 2
    id_grupo_viagem = quebra_de_viagem.cumsum()
    
    # 3. Criar o Case ID final
    # Combina o ID do ônibus com o ID do grupo
    # Ex: "HI858_1", "HI858_2", etc.
    df[CASE_ID] = df[COL_VEICULO] + '_' + id_grupo_viagem.astype(str)
    
    print("Coluna 'Case ID' criada com sucesso.")
    return df

def create_event_log_from_gps(df, stops_gdf, distance_threshold_meters=50):
    """
    Função principal que transforma pings de GPS em um log de eventos.
    
    Esta função implementa o "geofencing" descrito no seu guia.
    """
    
    # --- 1. Preparar Dados de GPS ---
    
    # Converte o DataFrame do pandas em um GeoDataFrame
    # para podermos fazer cálculos geoespaciais.
    gps_gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.points_from_xy(df[COL_LON], df[COL_LAT]),
        crs="EPSG:4326" # CRS padrão (WGS84) para Lat/Lon
    )
    
    # Como o seu guia menciona, precisamos garantir que o CRS seja o mesmo.
    # Vamos converter ambos para um CRS métrico (em metros) para que 
    # o 'distance_threshold_meters' funcione corretamente.
    
    # EPSG:32722 é o UTM 22S, bom para a região de Curitiba
    crs_metric = "EPSG:32722" 
    gps_gdf = gps_gdf.to_crs(crs_metric)
    stops_gdf = stops_gdf.to_crs(crs_metric)

    print(f"Convertendo GPS e Pontos para o CRS {crs_metric}...")
    
    # --- 2. Junção Espacial (Spatial Join) ---
    
    # Esta é a mágica do geofencing.
    # Para cada ping de GPS, encontra o ponto de ônibus mais próximo
    # DESDE QUE esteja dentro do 'distance_threshold_meters'.
    
    event_log_raw = gpd.sjoin_nearest(
        gps_gdf,
        stops_gdf, # 'stops_gdf' é o GeoDataFrame dos pontos de ônibus
        max_distance=distance_threshold_meters,
        how='left', # Mantém todos os pings de GPS, mesmo os que não têm ponto próximo
        rsuffix='stop' # Renomeia colunas do ponto (ex: 'stop_name')
    )
    
    print("Junção espacial (sjoin_nearest) concluída.")
    
    # --- 3. Gerar Atividades (Chegada / Partida) ---
    
    # A lógica exata de "Chegada" e "Partida" (como no seu guia)
    # é complexa. Vamos começar com uma simplificação:
    # A atividade é "Passou no Ponto X".
    
    # Pega o nome do ponto (da junção) e preenche vazios
    # (pings longe de qualquer ponto)
    event_log_raw['stop_name'] = event_log_raw['stop_name'].fillna('EM_MOVIMENTO')
    
    # A atividade é o nome do ponto
    event_log_raw[ACTIVITY] = event_log_raw['stop_name']

    # O timestamp é o tempo do ping
    event_log_raw[TIMESTAMP] = event_log_raw[COL_TIMESTAMP_API]
    
    # --- 4. Limpar o Log de Eventos ---
    
    # Nós não queremos 50 eventos "EM_MOVIMENTO" seguidos.
    # Vamos manter apenas as linhas onde a atividade MUDOU.
    mudou_atividade = event_log_raw[ACTIVITY] != event_log_raw[ACTIVITY].shift(1)
    
    # Também queremos manter a primeira linha de cada novo Case ID
    mudou_case_id = event_log_raw[CASE_ID] != event_log_raw[CASE_ID].shift(1)
    
    # Filtra o DataFrame
    event_log_final = event_log_raw[mudou_atividade | mudou_case_id].copy()

    # Não precisamos mais de eventos "EM_MOVIMENTO"
    event_log_final = event_log_final[event_log_final[ACTIVITY] != 'EM_MOVIMENTO']
    
    # Seleciona apenas as colunas que o PM4Py precisa
    colunas_finais = [CASE_ID, ACTIVITY, TIMESTAMP, COL_VEICULO, COL_LINHA, COL_SENTIDO]
    event_log_final = event_log_final[colunas_finais]

    print("Log de eventos final gerado e limpo.")    
    return event_log_final